In [99]:
# Install PySpark
!pip install pyspark --quiet

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.streaming import *
import time
import os
from IPython.display import display, clear_output

# Create Spark Session with Structured Streaming support
spark = SparkSession.builder \
    .appName("Hospital_Patient_Monitoring_Streaming") \
    .master("local[*]") \
    .config("spark.sql.streaming.schemaInference", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created!")

Spark Session Created!


In [56]:
# Upload your patients_data_with_alerts.csv to Colab first (Files > Upload)
# Or use the direct link if hosted
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
from pyspark.sql.types import *
# Define schema (matches your CSV)
schema = StructType([
    #StructField("timestamp", TimestampType(), True),
    StructField("Patient Number", IntegerType(), True),
    StructField("Heart Rate (bpm)", IntegerType(), True),
    StructField("SpO2 Level (%)", IntegerType(), True),
    StructField("Systolic Blood Pressure (mmHg)", IntegerType(), True),
    StructField("Diastolic Blood Pressure (mmHg)", IntegerType(), True),
    StructField("Body Temperature (Â°C)", FloatType(), True),
    StructField("Fall Detection", StringType(), True),
    StructField("Predicted Disease", StringType(), True),
    StructField("Data Accuracy (%)", IntegerType(), True),
    StructField("Heart Rate Alert", StringType(), True),
    StructField("SpO2 Level Alert", StringType(), True),
    StructField("Blood Pressure Alert", StringType(), True),
    StructField("Temperature Alert", StringType(), True),
    StructField("timestamp", TimestampType(), True)# We'll add this for streaming
])

# Load full data and add timestamp for simulation
df_full = spark.read.csv("/content/drive/MyDrive/patients_data_with_alerts.csv", header=True, schema=schema)
# Add simulated timestamp for streaming demo
from pyspark.sql.functions import current_timestamp, monotonically_increasing_id, expr
df_full = df_full.withColumn("timestamp", current_timestamp() - expr("INTERVAL 1 HOURS") * (monotonically_increasing_id() % 1000))

print(f"Loaded {df_full.count()} records")
df_full.show(10)

Loaded 50000 records
+--------------+----------------+--------------+------------------------------+-------------------------------+----------------------+--------------+-----------------+-----------------+----------------+----------------+--------------------+-----------------+--------------------+
|Patient Number|Heart Rate (bpm)|SpO2 Level (%)|Systolic Blood Pressure (mmHg)|Diastolic Blood Pressure (mmHg)|Body Temperature (Â°C)|Fall Detection|Predicted Disease|Data Accuracy (%)|Heart Rate Alert|SpO2 Level Alert|Blood Pressure Alert|Temperature Alert|           timestamp|
+--------------+----------------+--------------+------------------------------+-------------------------------+----------------------+--------------+-----------------+-----------------+----------------+----------------+--------------------+-----------------+--------------------+
|             1|              96|            92|                           101|                             89|             36.904682|     

In [100]:
# Create watched directory for streaming simulation
stream_dir = "/content/stream_data"
os.makedirs(stream_dir, exist_ok=True)

# Split data into small batches for realistic streaming
batches = df_full.randomSplit([0.1] * 10)  # ~10 batches

for i, batch in enumerate(batches):
    batch_path = f"{stream_dir}/batch_{i:02d}.csv"
    batch.write.mode("overwrite").csv(batch_path, header=True)
    print(f"Written batch {i} with {batch.count()} records to {batch_path}")

print("Streaming data prepared!")

Written batch 0 with 5108 records to /content/stream_data/batch_00.csv
Written batch 1 with 5004 records to /content/stream_data/batch_01.csv
Written batch 2 with 5012 records to /content/stream_data/batch_02.csv
Written batch 3 with 4959 records to /content/stream_data/batch_03.csv
Written batch 4 with 5035 records to /content/stream_data/batch_04.csv
Written batch 5 with 4780 records to /content/stream_data/batch_05.csv
Written batch 6 with 5012 records to /content/stream_data/batch_06.csv
Written batch 7 with 5088 records to /content/stream_data/batch_07.csv
Written batch 8 with 4901 records to /content/stream_data/batch_08.csv
Written batch 9 with 5101 records to /content/stream_data/batch_09.csv
Streaming data prepared!


In [140]:
# Read stream from watched directory
streaming_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .schema(schema) \
    .load(stream_dir)

# Add watermark and timestamp processing
streaming_df = streaming_df.withColumn("timestamp", to_timestamp(col("timestamp")))

# Tumbling 2-minute window aggregation per patient
windowed_df = streaming_df \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window(col("timestamp"), "2 minutes"),
        col("Patient Number")
    ) \
    .agg(
        avg("Heart Rate (bpm)").alias("avg_heart_rate"),
        count("*").alias("readings")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "Patient Number",
        "avg_heart_rate",
        "readings"
    )

# Alert: Sustained elevated HR (>100 bpm)
alert_df = windowed_df.filter(col("avg_heart_rate") > 100)

# Console output for alerts
alert_query = alert_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime='5 seconds') \
    .start()

print("Streaming pipeline started! Watch for alerts...")



Streaming pipeline started! Watch for alerts...


In [102]:
print("Windows with Avg HR > 100")

spark.sql("""
SELECT *
FROM alert_table
ORDER BY `Patient Number`, window_start
""").show(100, truncate=False)

Windows with Avg HR > 100
+------------+----------+--------------+--------------+--------+
|window_start|window_end|Patient Number|avg_heart_rate|readings|
+------------+----------+--------------+--------------+--------+
+------------+----------+--------------+--------------+--------+



In [103]:
for q in spark.streams.active:
    print(q.name, q.isActive)

None True


In [104]:
spark.sql("SHOW TABLES").show(truncate=False)

+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|         |alert_table|true       |
+---------+-----------+-----------+



In [105]:
from pyspark.sql.functions import *

df = spark.read.csv(
    "/content/drive/MyDrive/patients_data_with_alerts.csv",
    header=True,
    inferSchema=True
)

df.select(
    max("Heart Rate (bpm)").alias("max_hr"),
    avg("Heart Rate (bpm)").alias("avg_hr")
).show()

+------+---------+
|max_hr|   avg_hr|
+------+---------+
|   149|104.50942|
+------+---------+



In [106]:
df.filter(col("Heart Rate (bpm)") > 100).show(20, truncate=False)

+--------------+----------------+--------------+------------------------------+-------------------------------+---------------------+--------------+-----------------+-----------------+----------------+----------------+--------------------+-----------------+
|Patient Number|Heart Rate (bpm)|SpO2 Level (%)|Systolic Blood Pressure (mmHg)|Diastolic Blood Pressure (mmHg)|Body Temperature (°C)|Fall Detection|Predicted Disease|Data Accuracy (%)|Heart Rate Alert|SpO2 Level Alert|Blood Pressure Alert|Temperature Alert|
+--------------+----------------+--------------+------------------------------+-------------------------------+---------------------+--------------+-----------------+-----------------+----------------+----------------+--------------------+-----------------+
|4             |137             |84            |167                           |97                             |37.0187665357031     |Yes           |Asthma           |99               |High            |Low             |High    

In [107]:
from pyspark.sql.functions import *

batch_windows = (
    df_full.withColumn("timestamp", to_timestamp(col("timestamp"))) # Changed df to df_full
      .groupBy(
          window("timestamp", "2 minutes"),
          col("Patient Number")
      )
      .agg(avg("Heart Rate (bpm)").alias("avg_hr"))
      .orderBy(desc("avg_hr"))
)

batch_windows.show(20, truncate=False)

+------------------------------------------+--------------+------+
|window                                    |Patient Number|avg_hr|
+------------------------------------------+--------------+------+
|{2026-05-28 22:14:00, 2026-05-28 22:16:00}|39406         |149.0 |
|{2026-06-02 06:14:00, 2026-06-02 06:16:00}|6302          |149.0 |
|{2026-05-06 14:14:00, 2026-05-06 14:16:00}|49942         |149.0 |
|{2026-05-31 09:14:00, 2026-05-31 09:16:00}|24347         |149.0 |
|{2026-06-03 11:14:00, 2026-06-03 11:16:00}|3273          |149.0 |
|{2026-06-09 19:14:00, 2026-06-09 19:16:00}|28121         |149.0 |
|{2026-06-04 03:14:00, 2026-06-04 03:16:00}|4257          |149.0 |
|{2026-05-16 10:14:00, 2026-05-16 10:16:00}|36706         |149.0 |
|{2026-06-05 04:14:00, 2026-06-05 04:16:00}|5232          |149.0 |
|{2026-05-19 22:14:00, 2026-05-19 22:16:00}|15622         |149.0 |
|{2026-06-08 10:14:00, 2026-06-08 10:16:00}|21154         |149.0 |
|{2026-05-26 20:14:00, 2026-05-26 20:16:00}|2456          |149

In [108]:
from pyspark.sql.functions import col

filtered_batch_windows = batch_windows.filter(col("avg_hr") > 100)
filtered_batch_windows.show(20, truncate=False)

+------------------------------------------+--------------+------+
|window                                    |Patient Number|avg_hr|
+------------------------------------------+--------------+------+
|{2026-05-28 22:14:00, 2026-05-28 22:16:00}|39406         |149.0 |
|{2026-06-02 06:14:00, 2026-06-02 06:16:00}|6302          |149.0 |
|{2026-05-06 14:14:00, 2026-05-06 14:16:00}|49942         |149.0 |
|{2026-05-31 09:14:00, 2026-05-31 09:16:00}|24347         |149.0 |
|{2026-06-03 11:14:00, 2026-06-03 11:16:00}|3273          |149.0 |
|{2026-06-09 19:14:00, 2026-06-09 19:16:00}|28121         |149.0 |
|{2026-06-04 03:14:00, 2026-06-04 03:16:00}|4257          |149.0 |
|{2026-05-16 10:14:00, 2026-05-16 10:16:00}|36706         |149.0 |
|{2026-06-05 04:14:00, 2026-06-05 04:16:00}|5232          |149.0 |
|{2026-05-19 22:14:00, 2026-05-19 22:16:00}|15622         |149.0 |
|{2026-06-08 10:14:00, 2026-06-08 10:16:00}|21154         |149.0 |
|{2026-05-26 20:14:00, 2026-05-26 20:16:00}|2456          |149

In [133]:
# Stop any existing streaming queries named 'alert_table' to avoid conflicts
for stream in spark.streams.active:
    if stream.name == "alert_table": # This assumes alert_query's name is alert_table from prev run
        print(f"Stopping existing stream: {stream.name}")
        stream.stop()
        stream.awaitTermination()

# Restart alert_table memory sink
alert_memory_query = alert_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("alert_table") \
    .trigger(processingTime='5 seconds') \
    .start()

print("alert_table stream restarted.")

alert_table stream restarted.


In [135]:
import time
time.sleep(15) # Give some more time for streams to process multiple files

In [136]:
print("Content of alert_table after restart and processing:")
spark.sql("SELECT * FROM alert_table").show(truncate=False)

Content of alert_table after restart and processing:
+------------+----------+--------------+--------------+--------+
|window_start|window_end|Patient Number|avg_heart_rate|readings|
+------------+----------+--------------+--------------+--------+
+------------+----------+--------------+--------------+--------+



In [139]:
# Save window results to a static DataFrame
window_results = (
    spark.read.csv(
        "/content/drive/MyDrive/patients_data_with_alerts.csv",
        header=True,
        inferSchema=True
    )
)

In [112]:
alert_df = windowed_df.filter(col("avg_heart_rate") > 100)

In [134]:
print("Rows in raw streaming aggregation:")

# Restart window_table memory sink
window_query = windowed_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("window_table") \
    .trigger(processingTime='5 seconds') \
    .start()

print("window_table stream restarted.")

Rows in raw streaming aggregation:
window_table stream restarted.


In [143]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Read the original dataset
df = spark.read.csv(
    "/content/drive/MyDrive/patients_data_with_alerts.csv",
    header=True,
    inferSchema=True
)

# Add simulated timestamp for further processing as the CSV does not contain one
df = df.withColumn("timestamp", current_timestamp() - expr("INTERVAL 1 HOURS") * (monotonically_increasing_id() % 1000))

# Create 2-minute tumbling windows
windowed_batch = (
    df.groupBy(
        window("timestamp", "2 minutes"),
        col("Patient Number")
    )
    .agg(
        avg("Heart Rate (bpm)").alias("avg_heart_rate")
    )
    .select(
        col("Patient Number"),
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "avg_heart_rate"
    )
)

# Previous window per patient
w = Window.partitionBy("Patient Number").orderBy("window_start")

clinical_alerts = (
    windowed_batch
    .withColumn(
        "prev_avg_hr",
        lag("avg_heart_rate").over(w)
    )
    .filter(
        (col("avg_heart_rate") > 100) &
        (col("prev_avg_hr") > 100)
    )
)

print("=== CLINICAL ALERTS ===")
clinical_alerts.show(truncate=False)


=== CLINICAL ALERTS ===
+--------------+------------+----------+--------------+-----------+
|Patient Number|window_start|window_end|avg_heart_rate|prev_avg_hr|
+--------------+------------+----------+--------------+-----------+
+--------------+------------+----------+--------------+-----------+



In [144]:
windowed_batch.orderBy(desc("avg_heart_rate")).show(20, truncate=False)

+--------------+-------------------+-------------------+--------------+
|Patient Number|window_start       |window_end         |avg_heart_rate|
+--------------+-------------------+-------------------+--------------+
|48599         |2026-05-20 22:18:00|2026-05-20 22:20:00|149.0         |
|37276         |2026-06-03 09:18:00|2026-06-03 09:20:00|149.0         |
|11273         |2026-06-03 12:18:00|2026-06-03 12:20:00|149.0         |
|38070         |2026-06-11 23:18:00|2026-06-11 23:20:00|149.0         |
|33193         |2026-06-06 20:18:00|2026-06-06 20:20:00|149.0         |
|27922         |2026-05-07 11:18:00|2026-05-07 11:20:00|149.0         |
|16034         |2026-06-13 11:18:00|2026-06-13 11:20:00|149.0         |
|32493         |2026-05-25 08:18:00|2026-05-25 08:20:00|149.0         |
|14033         |2026-06-13 12:18:00|2026-06-13 12:20:00|149.0         |
|22026         |2026-06-13 19:18:00|2026-06-13 19:20:00|149.0         |
|18274         |2026-06-03 11:18:00|2026-06-03 11:20:00|149.0   

In [150]:
windowed_batch.filter(
    col("Patient Number") == 48599
).orderBy("window_start").show(truncate=False)

+--------------+-------------------+-------------------+--------------+
|Patient Number|window_start       |window_end         |avg_heart_rate|
+--------------+-------------------+-------------------+--------------+
|48599         |2026-05-20 22:24:00|2026-05-20 22:26:00|149.0         |
+--------------+-------------------+-------------------+--------------+

